In [1]:
import pandas as pd
import joblib
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

print("--- Starting ML Course Interest Model Build (KNN) ---")

# 1. Load the dataset
try:
    df = pd.read_csv('../Datasets/Course_Recommendation/MOOC.csv', encoding='latin-1')
    print("'MOOC.csv' dataset loaded successfully!")
except FileNotFoundError:
    print("Error: 'MOOC.csv' not found.")
    exit()

# 2. Clean data
df.dropna(subset=['Course Name', 'all_skill'], inplace=True)
df.drop_duplicates(subset='Course Name', inplace=True)
df.reset_index(drop=True, inplace=True)

# 3. Define B.Tech interest categories and their keywords
INTEREST_KEYWORDS = {
    'data_analysis': ['excel', 'data analysis', 'sql', 'tableau', 'power bi', 'statistics'],
    'ai_ml': ['machine learning', 'ai', 'deep learning', 'tensorflow', 'pytorch', 'nlp', 'computer vision', 'data science'],
    'web_dev': ['html', 'css', 'javascript', 'react', 'angular', 'vue', 'node.js', 'web development'],
    'cloud': ['aws', 'azure', 'google cloud', 'gcp', 'devops', 'docker', 'kubernetes'],
    'management': ['project management', 'agile', 'scrum', 'product management', 'leadership']
}

# 4. Feature Engineering: Create a "Profile" for each course
print("\n--- Profiling all courses against interest categories ---")
course_profiles = []
for index, row in df.iterrows():
    skills_text = str(row['all_skill']).lower()
    profile = {
        'Course Name': row['Course Name'],
        'data_analysis': sum([1 for keyword in INTEREST_KEYWORDS['data_analysis'] if keyword in skills_text]),
        'ai_ml': sum([1 for keyword in INTEREST_KEYWORDS['ai_ml'] if keyword in skills_text]),
        'web_dev': sum([1 for keyword in INTEREST_KEYWORDS['web_dev'] if keyword in skills_text]),
        'cloud': sum([1 for keyword in INTEREST_KEYWORDS['cloud'] if keyword in skills_text]),
        'management': sum([1 for keyword in INTEREST_KEYWORDS['management'] if keyword in skills_text])
    }
    course_profiles.append(profile)

profile_df = pd.DataFrame(course_profiles)
print("All courses profiled.")

# 5. ML Model Training (KNN)
print("\n--- Training K-Nearest Neighbors Model ---")
# Separate features (the scores) from the labels (Course Name)
features = profile_df.drop(columns=['Course Name'])

# Scale features so that all interests are weighted equally
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# Train the NearestNeighbors model
knn_model = NearestNeighbors(n_neighbors=10, algorithm='brute', metric='cosine')
knn_model.fit(features_scaled)
print("KNN model trained successfully.")

# 6. Save the ML model, scaler, and course names
output_dir = '../Models/Course_Recommendation'
os.makedirs(output_dir, exist_ok=True)

joblib.dump(knn_model, os.path.join(output_dir, 'knn_model.joblib'))
joblib.dump(scaler, os.path.join(output_dir, 'knn_scaler.joblib'))
joblib.dump(profile_df['Course Name'], os.path.join(output_dir, 'knn_course_names.joblib')) # Save the names in the correct order

print(f"KNN Model, Scaler, and Course List saved in: {output_dir}")

--- Starting ML Course Interest Model Build (KNN) ---
✅ 'MOOC.csv' dataset loaded successfully!

--- Profiling all courses against interest categories ---
✅ All courses profiled.

--- Training K-Nearest Neighbors Model ---
✅ KNN model trained successfully.
✅ KNN Model, Scaler, and Course List saved in: ../Models/Course_Recommendation
